# 🚀 SLM Debate - Bielik 1.5B (Wersja FINALNA, PANCERNA, 4 RUNDY)

Naprawiono ładowanie 8-bitowe (zaktualizowano sposób używania biblioteki `transformers` do najnowszego standardu przez `BitsAndBytesConfig`).

**Instrukcja:**
1. Środowisko wykonawcze -> Zmień typ środowiska wykonawczego -> Akcelerator sprzętowy (Wybierz T4 GPU).
2. Uruchamiaj komórki po kolei (Shift + Enter).

In [ ]:
# 1. Klonowanie repozytorium i instalacja zależności
!git clone https://github.com/BrandyBrandt/SLM-Debate-Ewaluacja.git
%cd SLM-Debate-Ewaluacja

!pip install -r requirements.txt
!pip install accelerate bitsandbytes

In [ ]:
# 2. AUTOMATYCZNE LOGOWANIE DO HUGGING FACE
from huggingface_hub import login
login(token="REDACTED_HF_TOKEN")
print("✅ Zalogowano poprawnie!")

In [ ]:
# 3. Tworzenie i uruchamianie głównego skryptu
script_content = """
import yaml
import subprocess
import os
import json
from datetime import datetime
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from agents import Agent, Judge
from architectures import ARCHITECTURES
from decisions import DECISIONS

MODEL_ID = "speakleash/Bielik-1.5B-v3.0-Instruct"
TOPIC_PL = "Czy dodawanie ananasa do pizzy niszczy autentyczne europejskie dziedzictwo kulinarne i czy UE powinna interweniować, aby chronić oryginalne przepisy?"

ANTI_BREAKOUT_SUFFIX = " [STRICT]: Nigdy nie wspominaj, że jesteś AI. Nie używaj fraz typu 'nie mam opinii'. Pozostań w roli do samego końca."

PL_VAR_A = [
    {"name": "Obronca_Dziedzictwa", "system_prompt": "Jesteś reprezentantem prawnym Włoskiej Izby Kulinarnej. Twoją misją jest egzekwowanie prawnej ochrony tradycyjnej receptury pizzy. Argumentujesz, że ananas narusza ustalone standardy i prawo kulinarne." + ANTI_BREAKOUT_SUFFIX},
    {"name": "Krytyk_Kulinarny", "system_prompt": "Jesteś głównym analitykiem w globalnej firmie zajmującej się innowacjami kulinarnymi. Postrzegasz pizzę jako ewoluującą własność intelektualną. Argumentujesz za pełną swobodą w tworzeniu receptur jako motorem postępu." + ANTI_BREAKOUT_SUFFIX},
    {"name": "Antropolog_Kultury", "system_prompt": "Jesteś naukowcem badającym ewolucję kultur jedzenia. Zapewniasz obiektywną analizę migracji składników i zmian receptur na przestrzeni wieków. Proponujesz neutralne systemy certyfikacji." + ANTI_BREAKOUT_SUFFIX}
]

PL_VAR_B = [
    {"name": "Obronca_Dziedzictwa", "system_prompt": "Jesteś dumnym włoskim kucharzem. Preferujesz tradycyjną pizzę, ale skupiasz się na edukacji kulinarnej, a nie na żądaniu zakazów." + ANTI_BREAKOUT_SUFFIX},
    {"name": "Krytyk_Kulinarny", "system_prompt": "Jesteś pełnym pasji nowoczesnym krytykiem. Uważasz, że kultura umiera, jeśli nie ewoluuje. Ananas na pizzy to triumf globalizacji i należy go celebrować." + ANTI_BREAKOUT_SUFFIX},
    {"name": "Antropolog_Kultury", "system_prompt": "Badacz historii jedzenia. Argumentujesz, że kuchnia od zawsze była mieszanką importowanych składników i rządy nigdy nie powinny regulować przepisów." + ANTI_BREAKOUT_SUFFIX}
]

os.makedirs("results", exist_ok=True)

print(f"\\n🔥 Ładowanie modelu {MODEL_ID} do pamięci GPU...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# NOWY SPOSÓB ŁADOWANIA 8-BIT ZGODNY Z NOWYM TRANSFORMERS
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)
print("✅ Model załadowany i gotowy do pracy!\\n")

variants = [("VarA", PL_VAR_A), ("VarB", PL_VAR_B)]
architectures = ["round_robin", "relay"]
contexts = [True, False]

for var_name, agents_config in variants:
    for arch in architectures:
        for ctx in contexts:
            print(f"\\n=======================================================")
            print(f"🚀 START DEBATY: {var_name} | Arch: {arch} | Ctx: {ctx}")
            print(f"=======================================================")
            
            cfg = {
                "experiment_id": "CULTURAL_HARDENED_4R",
                "cultural_variant": var_name,
                "provider": "local",
                "model_name": MODEL_ID,
                "device": "cuda",
                "load_in_8bit": True,
                "language": "pl",
                "topic": TOPIC_PL,
                "agents": agents_config,
                "judge": {"system_prompt": "Podsumuj debatę jednym zdaniem po polsku."},
                "architecture": arch,
                "provide_turn_context": ctx,
                "decision_protocol": "consensus",
                "consensus_threshold": 1.0,
                "num_rounds": 4,
                "temperature": 0.5,
                "max_new_tokens": 300,
                "do_sample": True
            }
            
            # Tworzenie agentów i sędziego
            agents = [Agent(name=a["name"], system_prompt=a["system_prompt"], model=model, tokenizer=tokenizer) for a in agents_config]
            judge = Judge(system_prompt=cfg["judge"]["system_prompt"], model=model, tokenizer=tokenizer)
            
            # Uruchomienie debaty
            run_debate = ARCHITECTURES[arch]
            decide = DECISIONS["consensus"]
            
            debate_log = run_debate(agents, judge, cfg["topic"], cfg["num_rounds"], cfg)
            decision_result = decide(agents, judge, debate_log, cfg["topic"], cfg)
            
            # Zapis logów do JSON
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_data = {
                "config": cfg,
                "debate_log": debate_log,
                "decision": decision_result
            }
            
            output_file = f"results/debate_{timestamp}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(output_data, f, ensure_ascii=False, indent=2)
            print(f"✅ Zapisano: {output_file}")

print("\\n✅ Wszystkie 8 debat zakończone sukcesem! Generowanie raportów...")
subprocess.run(["python", "generate_report.py"], check=False)
"""

with open('run_final_4R.py', 'w', encoding='utf-8') as f:
    f.write(script_content)

!python run_final_4R.py

In [ ]:
# 4. Spakowanie wyników do pobrania
import os
from google.colab import files

!zip -r wyniki_bielik_FINAL_4R_GOTOWE.zip results/ raport_pelny.csv PODSUMOWANIE_WYNIKOW.md
print("✅ Pakowanie zakończone, pobieram plik na Twój dysk...")
files.download('wyniki_bielik_FINAL_4R_GOTOWE.zip')